# 第 23 章 OpenCV 物件追蹤

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import sys
from opencv_tools import opencv_tools # 匯入封裝的功能
from ultralytics import YOLO

### 23-2-2 ROI 框選：cv2.selectROI

In [2]:
cap = cv2.VideoCapture("sample/video/cat.mp4")
ret, frame = cap.read()  # 取得第一幀
bbox = cv2.selectROI("Select Object", frame, showCrosshair=False)

print("框選結果：", bbox)

cap.release()
cv2.destroyWindow("Select Object")

框選結果： (211, 50, 220, 213)


### 23-3-3 程式範例：透過 KCF 追蹤器追蹤物件

In [7]:
# 開啟影片並擷取第一幀供使用者框選目標
cap = cv2.VideoCapture("sample/video/cat.mp4")
ret, frame = cap.read()
bbox = cv2.selectROI("Select Object", frame, showCrosshair=False)
cv2.destroyWindow("Select Object")

# 初始化 KCF 追蹤器
tracker = cv2.legacy.TrackerKCF_create()
tracker.init(frame, bbox)

# 開始追蹤
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                      lineType=cv2.LINE_AA)

    cv2.imshow("KCF Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 23-4-3 程式範例：透過 CSRT 追蹤器追蹤物件

In [9]:
# 開啟影片並擷取第一幀供使用者框選目標
cap = cv2.VideoCapture("sample/video/cat.mp4")
ret, frame = cap.read()
bbox = cv2.selectROI("Select Object", frame, showCrosshair=False)
cv2.destroyWindow("Select Object")

# 初始化 CSRT 追蹤器
tracker = cv2.legacy.TrackerCSRT_create()
tracker.init(frame, bbox)

# 開始追蹤
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                      lineType=cv2.LINE_AA)

    cv2.imshow("CSRT Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 23-5-2 程式範例：「單次」偵測，並交由追蹤器接力

In [17]:
# --- 設定區 ---
TARGET_CLASS = "person"
CONF_THRESHOLD = 0.5

# --- 載入模型與影片 ---
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("sample/video/walking.mp4")

# --- 第一幀執行 YOLO 偵測並初始化追蹤器 ---
ret, frame = cap.read()
trackers = []

results = model(frame, conf=CONF_THRESHOLD, verbose=False)
for box in results[0].boxes:
    cls_name = results[0].names[int(box.cls[0])]
    if cls_name != TARGET_CLASS:
        continue

    x1, y1, x2, y2 = map(int, box.xyxy[0])
    # YOLOv8 回傳 xyxy 格式，需轉換為追蹤器接受的 (x, y, w, h) 格式
    bbox = (x1, y1, x2 - x1, y2 - y1)

    # 初始化 KCF 追蹤器
    tracker = cv2.legacy.TrackerKCF_create()
    tracker.init(frame, bbox)
    trackers.append(tracker)

# --- 後續幀交由追蹤器處理 ---
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    for i, tracker in enumerate(trackers):
        success, bbox = tracker.update(frame)
        if success:
            x, y, w, h = map(int, bbox)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                          lineType=cv2.LINE_AA)
            cv2.putText(frame, f"{TARGET_CLASS} {i+1}", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("YOLOv8 + KCF | Single Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 23-5-3 程式範例：「週期性」的重新偵測並自動追蹤

In [18]:
# --- 設定區 ---
TARGET_CLASS = "person"
CONF_THRESHOLD = 0.5
DETECT_INTERVAL = 30  # 每隔幾幀重新執行一次 YOLO 偵測

# --- 載入模型與影片 ---
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("sample/video/walking.mp4")

trackers = []
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    # 每隔 DETECT_INTERVAL 幀重新偵測，清空並重新初始化追蹤器
    if frame_count % DETECT_INTERVAL == 0 or frame_count == 1:
        trackers = []
        results = model(frame, conf=CONF_THRESHOLD, verbose=False)

        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            if cls_name != TARGET_CLASS:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            tracker = cv2.legacy.TrackerKCF_create()
            tracker.init(frame, (x1, y1, x2-x1, y2-y1))
            trackers.append(tracker)

    else:
        # 非偵測幀，直接更新所有追蹤器
        for i, tracker in enumerate(trackers):
            success, bbox = tracker.update(frame)
            if success:
                x, y, w, h = map(int, bbox)
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                              lineType=cv2.LINE_AA)
                cv2.putText(frame, f"{TARGET_CLASS} {i+1}", (x, y-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("YOLOv8 + KCF | Periodic Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 23-5-4 進階：透過 KCF「週期性」地持續追蹤「同一目標」

In [20]:
def compute_iou(bbox1, bbox2):
    # 將 (x, y, w, h) 轉換為 (x1, y1, x2, y2)
    x1, y1 = bbox1[0], bbox1[1]
    x2, y2 = bbox1[0] + bbox1[2], bbox1[1] + bbox1[3]
    x3, y3 = bbox2[0], bbox2[1]
    x4, y4 = bbox2[0] + bbox2[2], bbox2[1] + bbox2[3]

    # 計算交集區域
    inter_x1, inter_y1 = max(x1, x3), max(y1, y3)
    inter_x2, inter_y2 = min(x2, x4), min(y2, y4)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    # 計算聯集區域
    union_area = (x2-x1)*(y2-y1) + (x4-x3)*(y4-y3) - inter_area
    return inter_area / union_area if union_area > 0 else 0

In [26]:
# --- 設定區 ---
TARGET_CLASS = "person"
CONF_THRESHOLD = 0.5
DETECT_INTERVAL = 30
IOU_THRESHOLD = 0.3  # IoU 低於此值才視為新目標

# --- 載入模型與影片 ---
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("sample/video/walking.mp4")

trackers = []
tracker_ids = []  # 每個追蹤器對應的 ID
next_id = 1
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    if frame_count % DETECT_INTERVAL == 0 or frame_count == 1:
        results = model(frame, conf=CONF_THRESHOLD, verbose=False)
        new_bboxes = []

        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            if cls_name != TARGET_CLASS:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            new_bboxes.append((x1, y1, x2-x1, y2-y1))

        # === IoU 匹配邏輯 ===
        # 取得所有現有追蹤器的當前位置
        current_bboxes = []
        for tracker in trackers:
            success, bbox = tracker.update(frame)
            current_bboxes.append(bbox if success else None)

        # 對每個新偵測框進行目標匹配
        matched_old = set()
        new_trackers = []
        new_ids = []

        for new_bbox in new_bboxes:
            best_iou, best_idx = 0, -1
            for i, cur_bbox in enumerate(current_bboxes):
                if cur_bbox is None or i in matched_old:
                    continue
                iou = compute_iou(new_bbox, cur_bbox)
                if iou > best_iou:
                    best_iou, best_idx = iou, i

            if best_iou >= IOU_THRESHOLD:
                # 同一目標，重新初始化校正位置，保留原 ID
                trackers[best_idx].init(frame, new_bbox)
                new_trackers.append(trackers[best_idx])
                new_ids.append(tracker_ids[best_idx])
                matched_old.add(best_idx)
            else:
                # 新目標，建立新追蹤器並分配新 ID
                tracker = cv2.legacy.TrackerKCF_create()
                tracker.init(frame, new_bbox)
                new_trackers.append(tracker)
                new_ids.append(next_id)
                next_id += 1

        trackers = new_trackers
        tracker_ids = new_ids

    # 繪製所有追蹤器的當前位置（統一在迴圈最後處理）
    for i, tracker in enumerate(trackers):
        success, bbox = tracker.update(frame)
        if success:
            x, y, w, h = map(int, bbox)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                          lineType=cv2.LINE_AA)
            cv2.putText(frame, f"{TARGET_CLASS} {tracker_ids[i]}", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("YOLOv8 + KCF | Persistent ID Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 23-5-5 進階：透過 CSRT「週期性」的持續追蹤「同一目標」

In [32]:
# --- 設定區 ---
TARGET_CLASS = "person"
CONF_THRESHOLD = 0.5
DETECT_INTERVAL = 30
IOU_THRESHOLD = 0.3   # IoU 低於此值才視為新目標
FRAME_SKIP = 2        # 每 N 幀只處理 1 幀（跳幀加速）

# --- 載入模型與影片 ---
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("sample/video/walking.mp4")

trackers = []
tracker_ids = []   # 每個追蹤器對應的 ID
next_id = 1
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    # 跳幀處理：每 FRAME_SKIP 幀才處理一次，其餘幀直接跳過
    if frame_count % FRAME_SKIP != 0:
        continue

    if frame_count % DETECT_INTERVAL == 0 or frame_count == FRAME_SKIP:
        results = model(frame, conf=CONF_THRESHOLD, verbose=False)
        new_bboxes = []
        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            if cls_name != TARGET_CLASS:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            new_bboxes.append((x1, y1, x2-x1, y2-y1))

        # === IoU 匹配邏輯 ===
        # 取得所有現有追蹤器的當前位置
        current_bboxes = []
        for tracker in trackers:
            success, bbox = tracker.update(frame)
            current_bboxes.append(bbox if success else None)

        # 對每個新偵測框進行目標匹配
        matched_old = set()
        new_trackers = []
        new_ids = []

        for new_bbox in new_bboxes:
            best_iou, best_idx = 0, -1
            for i, cur_bbox in enumerate(current_bboxes):
                if cur_bbox is None or i in matched_old:
                    continue
                iou = compute_iou(new_bbox, cur_bbox)
                if iou > best_iou:
                    best_iou, best_idx = iou, i

            if best_iou >= IOU_THRESHOLD:
                # 同一目標，重新初始化校正位置，保留原 ID
                trackers[best_idx].init(frame, new_bbox)
                new_trackers.append(trackers[best_idx])
                new_ids.append(tracker_ids[best_idx])
                matched_old.add(best_idx)
            else:
                # 新目標，建立新追蹤器並分配新 ID
                tracker = cv2.legacy.TrackerKCF_create()
                tracker.init(frame, new_bbox)
                new_trackers.append(tracker)
                new_ids.append(next_id)
                next_id += 1

        trackers = new_trackers
        tracker_ids = new_ids

    # 繪製所有追蹤器的當前位置（統一在迴圈最後處理）
    for i, tracker in enumerate(trackers):
        success, bbox = tracker.update(frame)
        if success:
            x, y, w, h = map(int, bbox)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                          lineType=cv2.LINE_AA)
            cv2.putText(frame, f"{TARGET_CLASS} {tracker_ids[i]}", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("YOLOv8 + CSRT | Persistent ID Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 23-E-1 程式範例：即時攝影機 YOLOv8n + KCF 自動追蹤

In [33]:
# --- IoU 計算函式（沿用 23-5-4） ---
def compute_iou(bbox1, bbox2):
    x1, y1 = bbox1[0], bbox1[1]
    x2, y2 = bbox1[0] + bbox1[2], bbox1[1] + bbox1[3]
    x3, y3 = bbox2[0], bbox2[1]
    x4, y4 = bbox2[0] + bbox2[2], bbox2[1] + bbox2[3]

    inter_x1, inter_y1 = max(x1, x3), max(y1, y3)
    inter_x2, inter_y2 = min(x2, x4), min(y2, y4)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    union_area = (x2-x1)*(y2-y1) + (x4-x3)*(y4-y3) - inter_area
    return inter_area / union_area if union_area > 0 else 0

In [34]:
# --- 共用前處理函式（沿用 22-E） ---
def enhance_brightness(frame, alpha=1.2, beta=50):
    """調整影像的亮度與對比度，改善攝影機畫面偏暗的問題。"""
    return cv2.convertScaleAbs(frame, alpha=alpha, beta=beta)

In [42]:
# --- 設定區 ---
TARGET_CLASS = "person"
CONF_THRESHOLD = 0.5
DETECT_INTERVAL = 30
IOU_THRESHOLD = 0.3
TRAIL_LENGTH = 300   # 軌跡保留的最大長度（幀數）
FRAME_SKIP = 2       # 每 N 幀只處理 1 幀（跳幀加速）

# --- 載入模型與影片 ---
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("sample/video/walking.mp4")

trackers = []
tracker_ids = []
next_id = 1
frame_count = 0

trail_history = {}  # {tracker_id: [center1, center2, ...]}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    # 跳幀處理：每 FRAME_SKIP 幀才處理一次，其餘幀直接跳過
    if frame_count % FRAME_SKIP != 0:
        continue

    # 前處理：調亮（視測試需求決定是否啟用）
    frame = enhance_brightness(frame)

    if frame_count % DETECT_INTERVAL == 0 or frame_count == FRAME_SKIP:
        results = model(frame, conf=CONF_THRESHOLD, verbose=False)
        new_bboxes = []

        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            if cls_name != TARGET_CLASS:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            new_bboxes.append((x1, y1, x2-x1, y2-y1))

        # 取得所有現有追蹤器的當前位置
        current_bboxes = []
        for tracker in trackers:
            success, bbox = tracker.update(frame)
            current_bboxes.append(bbox if success else None)

        # IoU 匹配
        matched_old = set()
        new_trackers = []
        new_ids = []

        for new_bbox in new_bboxes:
            best_iou, best_idx = 0, -1
            for i, cur_bbox in enumerate(current_bboxes):
                if cur_bbox is None or i in matched_old:
                    continue
                iou = compute_iou(new_bbox, cur_bbox)
                if iou > best_iou:
                    best_iou, best_idx = iou, i

            if best_iou >= IOU_THRESHOLD:
                trackers[best_idx].init(frame, new_bbox)
                new_trackers.append(trackers[best_idx])
                new_ids.append(tracker_ids[best_idx])
                matched_old.add(best_idx)
            else:
                tracker = cv2.legacy.TrackerKCF_create()
                tracker.init(frame, new_bbox)
                new_trackers.append(tracker)
                new_ids.append(next_id)
                next_id += 1

        trackers = new_trackers
        tracker_ids = new_ids

    # 繪製階段：邊界框、編號、軌跡
    for i, tracker in enumerate(trackers):
        success, bbox = tracker.update(frame)
        if success:
            x, y, w, h = map(int, bbox)
            tid = tracker_ids[i]

            # 取追蹤器邊界框的上 1/3 位置作為軌跡點（接近人物的胸部位置）
            center = (x + w // 2, y + h // 3)
            if tid not in trail_history:
                trail_history[tid] = []
            trail_history[tid].append(center)

            # 限制軌跡長度
            if len(trail_history[tid]) > TRAIL_LENGTH:
                trail_history[tid].pop(0)

            # 繪製軌跡（紅線）
            if len(trail_history[tid]) > 1:
                pts = np.array(trail_history[tid], dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [pts], isClosed=False,
                              color=(0, 0, 255), thickness=2,
                              lineType=cv2.LINE_AA)

            # 繪製邊界框與編號（綠色）
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                          lineType=cv2.LINE_AA)
            cv2.putText(frame, f"{TARGET_CLASS} {tid}", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("YOLOv8 + KCF | Trail Visualization", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()